# Clasificación de Perros y Gatos con CNN

En este cuaderno aprenderemos a clasificar imágenes de perros y gatos utilizando Redes Neuronales Convolucionales (CNN) y técnicas de **Data Augmentation**.

Este ejemplo está diseñado para ser muy similar a los que ya hemos visto de MNIST, pero aplicado a imágenes reales de colores y tamaños variados.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

## 1. Descarga y Preparación de Datos

Usaremos el dataset `cats_vs_dogs`. A diferencia de MNIST, estas imágenes no tienen el mismo tamaño, por lo que debemos redimensionarlas.

**¿Por qué no basta con usar una capa `layers.Resizing` en el bloque de preprocesamiento?**
Para entrenar, agrupamos las imágenes en lotes (**batches**). Si las fotos tienen tamaños distintos, TensorFlow no puede "apilarlas" en un bloque uniforme, lo que da un error de dimensiones antes siquiera de que el modelo empiece a procesarlas. Por eso, es obligatorio redimensionarlas **antes** de agruparlas con `.map()`.

**Entonces, ¿para qué sirve la capa `layers.Resizing`?**
Es muy útil cuando desplegamos el modelo (por ejemplo, en nuestra API). Al incluirla dentro del modelo, este se vuelve "autónomo": cualquier imagen que le enviemos, sea del tamaño que sea, será redimensionada automáticamente por la propia red antes de la primera convolución, sin que tengamos que programar ese ajuste fuera.

In [ ]:
IMG_SIZE = 180

def format_example(image, label):
    # Redimensiona una sola imagen (necesario antes del batching)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

# Descarga el dataset con sus etiquetas
(train_ds, val_ds, test_ds), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

# Unifica el tamaño de todas las imágenes
train_ds = train_ds.map(format_example)
val_ds = val_ds.map(format_example)
test_ds = test_ds.map(format_example)

# Procesa lotes (batch), mezcla y pre-carga para mayor velocidad
BATCH_SIZE = 32
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 2. Definición del Pipeline de Preprocesamiento y Augmentation

Integrar estas capas directamente en el modelo es la forma más limpia y moderna de trabajar en Keras.

In [ ]:
data_augmentation = models.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

## 3. Construcción del Modelo

Combinamos las capas de preprocesamiento, augmentation y la arquitectura CNN.

In [ ]:
model = models.Sequential([
  # Capa de entrada (acepta cualquier tamaño para redimensionar después)
  layers.Input(shape=(None, None, 3)),
  
  # 1. Preprocesamiento (Redimensionado autónomo, Aumento y Normalización)
  layers.Resizing(IMG_SIZE, IMG_SIZE),
  data_augmentation,
  layers.Rescaling(1./255),
  
  # 2. Capas Convolucionales
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  
  # 3. Clasificador (Capas Densas)
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dense(1, activation='sigmoid') # 0: Gato, 1: Perro
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

## 4. Entrenamiento

Al haber redimensionado las imágenes previamente, el entrenamiento ahora será fluido.

In [ ]:
epochs = 10
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

## 5. Evaluación y Guardado

In [ ]:
# Guardar el modelo para usarlo en la API
model.save('modelo_perros_gatos.keras')
print("Modelo exportado correctamente.")